# HW10-11 – компьютерное зрение в PyTorch

TODO: реализуйте требования из `homeworks/S10/S10-homework.md` (части S10 и S11) и заполните артефакты в `homeworks/HW10-11/artifacts/`.


## Быстрый чеклист
- Часть A (S10): CNN + аугментации + transfer learning на `ResNet18` (C1–C4) и выбор по `best_val_accuracy`.
- Часть B (S11): detection **или** segmentation (V1–V2), визуализация и базовая метрика.
- Результаты: `artifacts/runs.csv` и картинки в `artifacts/figures/`.


In [2]:
import os
import json
import csv
import random
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

import torchvision
import torchvision.transforms as T
import torchvision.models as models
from torchvision.datasets import STL10
from torchvision.models import ResNet18_Weights

from torchvision.transforms.functional import to_pil_image

FAST_DEV_RUN = True
SEED = 42

_cwd = os.getcwd()
if os.path.basename(os.path.normpath(_cwd)) == "HW10-11":
    _HW_ROOT = _cwd
else:
    _HW_ROOT = os.path.join(_cwd, "homeworks", "HW10-11")

DATA_ROOT = os.path.join(_HW_ROOT, "data")
ARTIFACT_DIR = os.path.join(_HW_ROOT, "artifacts")
FIG_DIR = os.path.join(ARTIFACT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

IMG_SIZE = 96

def accuracy_from_logits(logits, y):
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    running_acc = 0.0
    n = 0

    for batch_idx, (x, y) in enumerate(loader):
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        bs = x.size(0)
        running_loss += loss.item() * bs
        running_acc += accuracy_from_logits(logits.detach(), y) * bs
        n += bs

        if FAST_DEV_RUN and batch_idx >= 10:
            break

    return running_loss / max(1, n), running_acc / max(1, n)


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_acc = 0.0
    n = 0
    with torch.no_grad():
        for batch_idx, (x, y) in enumerate(loader):
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            loss = criterion(logits, y)

            bs = x.size(0)
            running_loss += loss.item() * bs
            running_acc += accuracy_from_logits(logits, y) * bs
            n += bs

            if FAST_DEV_RUN and batch_idx >= 10:
                break

    return running_loss / max(1, n), running_acc / max(1, n)



train_full = STL10(root=DATA_ROOT, split="train", download=True)

if FAST_DEV_RUN:
    all_idx = list(range(len(train_full)))
    rng = np.random.default_rng(SEED)
    rng.shuffle(all_idx)
    train_keep = all_idx[:2500]
    val_keep = all_idx[2500:3500]
    test_keep = list(range(0, 800))
else:
    train_keep = None
    val_keep = None
    test_keep = None

indices = np.arange(len(train_full))
perm = np.random.default_rng(SEED).permutation(indices)

if FAST_DEV_RUN:
    train_indices = np.array(train_keep)
    val_indices = np.array(val_keep)
else:
    val_size = int(0.2 * len(train_full))
    val_indices = perm[:val_size]
    train_indices = perm[val_size:]

train_base_indices = train_indices.tolist()
val_base_indices = val_indices.tolist()


def make_subset(dataset, indices):
    return torch.utils.data.Subset(dataset, indices)


base_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

aug_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomCrop(IMG_SIZE, padding=8),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

resnet_transform = base_transform


class STL10WithTransform(torch.utils.data.Dataset):
    def __init__(self, base_dataset, indices, transform):
        self.base = base_dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        x, y = self.base[self.indices[idx]]
        x = self.transform(x)
        return x, y


num_classes = 10


class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.net(x)
        x = x.flatten(1)
        return self.fc(x)


@dataclass
class ExperimentResult:
    experiment_id: str
    best_val_accuracy: float
    test_accuracy: float
    epochs_trained: int
    history: dict
    model_state_dict: dict
    model_summary: str
    best_val_loss: float


stl_test = STL10(root=DATA_ROOT, split="test", download=True)
if FAST_DEV_RUN:
    test_indices = test_keep
else:
    test_indices = list(range(len(stl_test)))

stl_test_ds = STL10WithTransform(stl_test, test_indices, base_transform)


BATCH_SIZE = 64 if not FAST_DEV_RUN else 32
NUM_WORKERS = 0

criterion = nn.CrossEntropyLoss()

results_rows = []

best_overall = None

best_curves = None
best_fig_path = os.path.join(FIG_DIR, "classification_curves_best.png")


def log_and_store(exp_id, task, dataset_name, seed, model_summary, optimizer_name, lr, epochs_trained, best_val_acc, test_acc, precision, recall, mean_iou, notes):
    row = {
        "experiment_id": exp_id,
        "task": task,
        "dataset": dataset_name,
        "seed": seed,
        "model_summary": model_summary,
        "optimizer": optimizer_name,
        "lr": lr,
        "epochs_trained": epochs_trained,
        "best_val_accuracy": best_val_acc,
        "test_accuracy": test_acc,
        "precision": precision,
        "recall": recall,
        "mean_iou": mean_iou,
        "notes": notes,
    }
    results_rows.append(row)


def run_simple_cnn(experiment_id, transform, lr=1e-3, epochs=3):
    train_ds = STL10WithTransform(train_full, train_base_indices, transform)
    val_ds = STL10WithTransform(train_full, val_base_indices, base_transform)

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = torch.utils.data.DataLoader(stl_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    model = SimpleCNN(num_classes=num_classes).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_val_acc = -1.0
    best_val_loss = float("inf")
    best_state = None
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(epochs):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        va_loss, va_acc = evaluate(model, val_loader, criterion, DEVICE)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_val_loss = va_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    test_loss, test_acc = evaluate(model, test_loader, criterion, DEVICE)

    model_summary = "SimpleCNN"
    log_and_store(
        exp_id=experiment_id,
        task="classification",
        dataset_name="STL10",
        seed=SEED,
        model_summary=model_summary,
        optimizer_name=optimizer.__class__.__name__,
        lr=lr,
        epochs_trained=epochs,
        best_val_acc=best_val_acc,
        test_acc=test_acc,
        precision="",
        recall="",
        mean_iou="",
        notes="FAST_DEV_RUN=%s" % FAST_DEV_RUN,
    )

    return ExperimentResult(
        experiment_id=experiment_id,
        best_val_accuracy=best_val_acc,
        test_accuracy=test_acc,
        epochs_trained=epochs,
        history=history,
        model_state_dict=best_state,
        model_summary=model_summary,
        best_val_loss=best_val_loss,
    )


def run_resnet18(experiment_id, mode, lr=1e-4, epochs=2):
    train_ds = STL10WithTransform(train_full, train_base_indices, resnet_transform)
    val_ds = STL10WithTransform(train_full, val_base_indices, resnet_transform)

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = torch.utils.data.DataLoader(stl_test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    try:
        backbone = models.resnet18(weights=ResNet18_Weights.DEFAULT)
    except TypeError:
        backbone = models.resnet18(pretrained=True)

    in_features = backbone.fc.in_features
    backbone.fc = nn.Linear(in_features, num_classes)

    if mode == "head_only":
        for p in backbone.parameters():
            p.requires_grad = False
        for p in backbone.fc.parameters():
            p.requires_grad = True
    elif mode == "partial_finetune":
        for p in backbone.parameters():
            p.requires_grad = False
        for p in backbone.layer4.parameters():
            p.requires_grad = True
        for p in backbone.fc.parameters():
            p.requires_grad = True
    else:
        raise ValueError("Unknown mode")

    model = backbone.to(DEVICE)

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(trainable_params, lr=lr)

    best_val_acc = -1.0
    best_state = None
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    for epoch in range(epochs):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        va_loss, va_acc = evaluate(model, val_loader, criterion, DEVICE)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    _, test_acc = evaluate(model, test_loader, criterion, DEVICE)

    model_summary = "resnet18(%s)" % mode

    log_and_store(
        exp_id=experiment_id,
        task="classification",
        dataset_name="STL10",
        seed=SEED,
        model_summary=model_summary,
        optimizer_name=optimizer.__class__.__name__,
        lr=lr,
        epochs_trained=epochs,
        best_val_acc=best_val_acc,
        test_acc=test_acc,
        precision="",
        recall="",
        mean_iou="",
        notes="FAST_DEV_RUN=%s" % FAST_DEV_RUN,
    )

    return ExperimentResult(
        experiment_id=experiment_id,
        best_val_accuracy=best_val_acc,
        test_accuracy=test_acc,
        epochs_trained=epochs,
        history=history,
        model_state_dict=best_state,
        model_summary=model_summary,
        best_val_loss=float("nan"),
    )


EPOCHS_CNN = 3 if not FAST_DEV_RUN else 1
EPOCHS_RESNET = 2 if not FAST_DEV_RUN else 1

exp_results = []

exp_results.append(run_simple_cnn("C1", transform=base_transform, lr=1e-3, epochs=EPOCHS_CNN))
exp_results.append(run_simple_cnn("C2", transform=aug_transform, lr=1e-3, epochs=EPOCHS_CNN))
exp_results.append(run_resnet18("C3", mode="head_only", lr=1e-4, epochs=EPOCHS_RESNET))
exp_results.append(run_resnet18("C4", mode="partial_finetune", lr=1e-4, epochs=EPOCHS_RESNET))

best_overall = max(exp_results, key=lambda r: r.best_val_accuracy)

best_model_path = os.path.join(ARTIFACT_DIR, "best_classifier.pt")
best_cfg_path = os.path.join(ARTIFACT_DIR, "best_classifier_config.json")

torch.save(best_overall.model_state_dict, best_model_path)

best_cfg = {
    "seed": SEED,
    "dataset": "STL10",
    "experiment_id": best_overall.experiment_id,
    "model_summary": best_overall.model_summary,
    "FAST_DEV_RUN": FAST_DEV_RUN,
    "IMG_SIZE": IMG_SIZE,
}

with open(best_cfg_path, "w", encoding="utf-8") as f:
    json.dump(best_cfg, f, ensure_ascii=False, indent=2)

best_curves = best_overall.history
plt.figure(figsize=(7, 4))
plt.plot(best_curves["train_loss"], label="train_loss")
plt.plot(best_curves["val_loss"], label="val_loss")
plt.title(f"Best model {best_overall.experiment_id}: Loss")
plt.legend()
plt.tight_layout()
plt.savefig(best_fig_path, dpi=150)
plt.close()

plt.figure(figsize=(7, 4))
ids = [r.experiment_id for r in exp_results]
vals = [r.best_val_accuracy for r in exp_results]
plt.bar(ids, vals)
plt.title("Classification compare (best val acc)")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "classification_compare.png"), dpi=150)
plt.close()

if FAST_DEV_RUN:
    sample_indices = list(range(8))
else:
    sample_indices = list(range(12))

class SimpleSTLWrapper(torch.utils.data.Dataset):
    def __init__(self, base_dataset, indices, transform):
        self.base = base_dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        x, _ = self.base[self.indices[idx]]
        return self.transform(x)

preview_ds = SimpleSTLWrapper(train_full, sample_indices, aug_transform)

loader = torch.utils.data.DataLoader(preview_ds, batch_size=len(sample_indices), shuffle=False)
x_aug = next(iter(loader))

mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
x_vis = x_aug * std + mean
x_vis = torch.clamp(x_vis, 0, 1)

cols = 4
rows = int(np.ceil(len(sample_indices) / cols))
plt.figure(figsize=(10, 3 * rows))
for i in range(len(sample_indices)):
    plt.subplot(rows, cols, i + 1)
    plt.imshow(to_pil_image(x_vis[i]))
    plt.axis("off")
    plt.title(f"aug {i}")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "augmentations_preview.png"), dpi=150)
plt.close()

runs_path = os.path.join(ARTIFACT_DIR, "runs.csv")
fieldnames = [
    "experiment_id",
    "task",
    "dataset",
    "seed",
    "model_summary",
    "optimizer",
    "lr",
    "epochs_trained",
    "best_val_accuracy",
    "test_accuracy",
    "precision",
    "recall",
    "mean_iou",
    "notes",
]

with open(runs_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results_rows)


from torchvision.datasets import OxfordIIITPet

try:
    seg_ds = OxfordIIITPet(root=DATA_ROOT, split="test", target_types=("segmentation",), download=True)
except TypeError:
    seg_ds = OxfordIIITPet(root=DATA_ROOT, split="test", target_types="segmentation", download=True)

if FAST_DEV_RUN:
    seg_indices = list(range(25))
else:
    seg_indices = list(range(len(seg_ds)))

seg_subset = torch.utils.data.Subset(seg_ds, seg_indices)

weights_seg = models.segmentation.FCN_ResNet50_Weights.DEFAULT
seg_model = models.segmentation.fcn_resnet50(weights=weights_seg)
seg_model.to(DEVICE)
seg_model.eval()

seg_categories = weights_seg.meta.get("categories", [])
cat_idx = None
dog_idx = None

for i, name in enumerate(seg_categories):
    if name == "cat":
        cat_idx = i
    if name == "dog":
        dog_idx = i

if cat_idx is None or dog_idx is None:
    cat_idx = 17
    dog_idx = 18

background_idx = 0

preprocess = weights_seg.transforms()

def seg_collate_fn(batch):
    images = [b[0] for b in batch]
    targets = [b[1] for b in batch]
    return images, targets

seg_loader = torch.utils.data.DataLoader(seg_subset, batch_size=2, shuffle=False, num_workers=0, collate_fn=seg_collate_fn)


def mask_to_fg_bool(mask):
    if isinstance(mask, torch.Tensor):
        m = mask
    else:
        m = T.PILToTensor()(mask)
    if m.ndim == 3:
        m = m[0]
    return (m > 0)


@torch.no_grad()
def infer_segmentation_v1_v2(images, targets, threshold_fg=0.7):

    ious_v1, prec_v1, rec_v1 = [], [], []
    ious_v2, prec_v2, rec_v2 = [], [], []

    pred_v1_fg_list = []
    pred_v2_fg_list = []

    def compute_metrics(pred_fg, gt_fg):
        pred_fg = pred_fg.bool()
        gt_fg = gt_fg.bool()

        inter = (pred_fg & gt_fg).sum().item()
        union = (pred_fg | gt_fg).sum().item()

        if union == 0:
            iou = 1.0 if inter == 0 else 0.0
        else:
            iou = inter / (union + 1e-9)

        tp = (pred_fg & gt_fg).sum().item()
        fp = (pred_fg & ~gt_fg).sum().item()
        fn = (~pred_fg & gt_fg).sum().item()

        precision = tp / (tp + fp + 1e-9)
        recall = tp / (tp + fn + 1e-9)
        return iou, precision, recall

    for img, t in zip(images, targets):
        img_t = preprocess(img).to(DEVICE)
        out = seg_model(img_t.unsqueeze(0))["out"]
        probs = torch.softmax(out, dim=1)[0]

        pred_argmax = probs.argmax(dim=0)
        pred_v1_fg = pred_argmax != background_idx

        p_fg = probs[cat_idx, :, :] + probs[dog_idx, :, :]
        pred_v2_fg = p_fg > threshold_fg

        gt_fg = mask_to_fg_bool(t)

        H, W = pred_argmax.shape[-2:]
        if gt_fg.shape[-2:] != (H, W):
            gt_t = gt_fg.unsqueeze(0).unsqueeze(0).float()
            gt_r = torch.nn.functional.interpolate(gt_t, size=(H, W), mode="nearest").squeeze()
            gt_fg = gt_r.bool()

        iou1, p1, r1 = compute_metrics(pred_v1_fg, gt_fg)
        iou2, p2, r2 = compute_metrics(pred_v2_fg, gt_fg)

        ious_v1.append(iou1)
        prec_v1.append(p1)
        rec_v1.append(r1)

        ious_v2.append(iou2)
        prec_v2.append(p2)
        rec_v2.append(r2)

        pred_v1_fg_list.append(pred_v1_fg.detach().cpu())
        pred_v2_fg_list.append(pred_v2_fg.detach().cpu())

    mean_iou_v1 = float(np.mean(ious_v1))
    mean_iou_v2 = float(np.mean(ious_v2))
    precision_v1 = float(np.mean(prec_v1))
    recall_v1 = float(np.mean(rec_v1))
    precision_v2 = float(np.mean(prec_v2))
    recall_v2 = float(np.mean(rec_v2))

    return {
        "mean_iou_v1": mean_iou_v1,
        "mean_iou_v2": mean_iou_v2,
        "precision_v1": precision_v1,
        "recall_v1": recall_v1,
        "precision_v2": precision_v2,
        "recall_v2": recall_v2,
        "pred_v1_fg": pred_v1_fg_list,
        "pred_v2_fg": pred_v2_fg_list,
    }


all_iou_v1 = []
all_iou_v2 = []
all_prec_v1 = []
all_prec_v2 = []
all_rec_v1 = []
all_rec_v2 = []

vis_examples = []

for batch_idx, (images, targets) in enumerate(seg_loader):
    out_metrics = infer_segmentation_v1_v2(images, targets, threshold_fg=0.7)

    all_iou_v1.append(out_metrics["mean_iou_v1"])
    all_iou_v2.append(out_metrics["mean_iou_v2"])
    all_prec_v1.append(out_metrics["precision_v1"])
    all_prec_v2.append(out_metrics["precision_v2"])
    all_rec_v1.append(out_metrics["recall_v1"])
    all_rec_v2.append(out_metrics["recall_v2"])

    if batch_idx == 0:
        vis_examples.append((images, targets, out_metrics["pred_v1_fg"], out_metrics["pred_v2_fg"]))

    if FAST_DEV_RUN and batch_idx >= 5:
        break

mean_iou_v1 = float(np.mean(all_iou_v1))
mean_iou_v2 = float(np.mean(all_iou_v2))
precision_v1 = float(np.mean(all_prec_v1))
recall_v1 = float(np.mean(all_rec_v1))
precision_v2 = float(np.mean(all_prec_v2))
recall_v2 = float(np.mean(all_rec_v2))

images0, targets0, pred_v1_fg0, pred_v2_fg0 = vis_examples[0]


n_vis = min(6, len(images0))
cols = 3
rows = n_vis

plt.figure(figsize=(10, 3 * rows))
for i in range(n_vis):
    plt.subplot(rows, cols, i * cols + 1)
    plt.imshow(images0[i])
    plt.axis("off")
    plt.title("img")

    plt.subplot(rows, cols, i * cols + 2)
    plt.imshow(pred_v1_fg0[i].numpy(), cmap="gray")
    plt.axis("off")
    plt.title("V1 fg")

    plt.subplot(rows, cols, i * cols + 3)
    plt.imshow(pred_v2_fg0[i].numpy(), cmap="gray")
    plt.axis("off")
    plt.title("V2 fg")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "segmentation_examples.png"), dpi=150)
plt.close()

plt.figure(figsize=(6, 4))
plt.bar(["V1", "V2"], [mean_iou_v1, mean_iou_v2], color=["tab:blue", "tab:orange"])
plt.ylabel("mean IoU (foreground)")
plt.title("Segmentation metrics")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "segmentation_metrics.png"), dpi=150)
plt.close()

log_and_store(
    exp_id="V1",
    task="segmentation",
    dataset_name="OxfordIIITPet",
    seed=SEED,
    model_summary=f"{type(seg_model).__name__} (fcn_resnet50 COCO) - argmax fg",
    optimizer_name="",
    lr="",
    epochs_trained=0,
    best_val_acc="",
    test_acc="",
    precision=precision_v1,
    recall=recall_v1,
    mean_iou=mean_iou_v1,
    notes=f"threshold_fg=0.3 unused (V1 uses argmax), FAST_DEV_RUN={FAST_DEV_RUN}",
)

log_and_store(
    exp_id="V2",
    task="segmentation",
    dataset_name="OxfordIIITPet",
    seed=SEED,
    model_summary=f"{type(seg_model).__name__} (fcn_resnet50 COCO) - p_fg threshold",
    optimizer_name="",
    lr="",
    epochs_trained=0,
    best_val_acc="",
    test_acc="",
    precision=precision_v2,
    recall=recall_v2,
    mean_iou=mean_iou_v2,
    notes=f"threshold_fg=0.7, FAST_DEV_RUN={FAST_DEV_RUN}",
)

with open(runs_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results_rows)

print("Finished HW10-11.")
print("Best classifier:", best_overall.experiment_id, "best_val_accuracy=", best_overall.best_val_accuracy, "test_accuracy=", best_overall.test_accuracy)
print("Segmentation: V1 mean_iou=", mean_iou_v1, "V2 mean_iou=", mean_iou_v2)


Finished HW10-11.
Best classifier: C4 best_val_accuracy= 0.5028409090909091 test_accuracy= 0.5284090909090909
Segmentation: V1 mean_iou= 0.394625786346747 V2 mean_iou= 0.31571366206317625
